# Check image-to-label pairs

This notebook checks whether every  image in  and  has a same-named YOLO  label. It also finds label files that do not have an image.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Works when the notebook is launched from either the repository root
# or the notebooks directory. Change this if your dataset is elsewhere.
DATASET_ROOT = Path('detection_dataset')
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path('../detection_dataset')
DATASET_ROOT = DATASET_ROOT.resolve()

SPLITS = ('train', 'test')
print(f'Dataset: {DATASET_ROOT}')

Dataset: /home/mehedinaeem/Desktop/Code/Bitol_Computer_Vision_System/detection_dataset


In [4]:
def check_split(dataset_root: Path, split: str):
    image_dir = dataset_root / 'images' / split
    label_dir = dataset_root / 'labels' / split

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(
            f'Missing directory for {split}: {image_dir} or {label_dir}'
        )

    images = {p.stem: p for p in image_dir.glob('*.jpg')}
    labels = {p.stem: p for p in label_dir.glob('*.txt') if p.name != 'classes.txt'}
    all_stems = sorted(images.keys() | labels.keys())

    rows = []
    for stem in all_stems:
        image = images.get(stem)
        label = labels.get(stem)
        if image and label:
            status = 'matched'
        elif image:
            status = 'missing_label'
        else:
            status = 'missing_image'

        rows.append({
            'split': split,
            'stem': stem,
            'image_file': image.name if image else None,
            'label_file': label.name if label else None,
            'status': status,
        })

    return pd.DataFrame(rows)

pair_report = pd.concat(
    [check_split(DATASET_ROOT, split) for split in SPLITS],
    ignore_index=True,
)
pair_report.head()

,split,stem,image_file,label_file,status
0,train,healthy_0001,healthy_0001.jpg,healthy_0001.txt,matched
1,train,healthy_0002,healthy_0002.jpg,healthy_0002.txt,matched
2,train,healthy_0003,healthy_0003.jpg,healthy_0003.txt,matched
3,train,healthy_0004,healthy_0004.jpg,healthy_0004.txt,matched
4,train,healthy_0005,healthy_0005.jpg,healthy_0005.txt,matched


In [5]:
summary = (
    pair_report.groupby('split', sort=False)
    .agg(
        jpg_images=('image_file', 'count'),
        txt_labels=('label_file', 'count'),
        matched_pairs=('status', lambda s: (s == 'matched').sum()),
        missing_labels=('status', lambda s: (s == 'missing_label').sum()),
        orphan_labels=('status', lambda s: (s == 'missing_image').sum()),
    )
    .reindex(SPLITS)
)
summary['all_images_have_labels'] = summary['missing_labels'].eq(0)
summary['all_labels_have_images'] = summary['orphan_labels'].eq(0)
display(summary)

,jpg_images,txt_labels,matched_pairs,missing_labels,orphan_labels,all_images_have_labels,all_labels_have_images
split,,,,,,,
train,1946,1946,1946,0,0,True,True
test,485,485,485,0,0,True,True


In [6]:
problems = pair_report[pair_report['status'] != 'matched'].reset_index(drop=True)

if problems.empty:
    print('OK: every .jpg image has a matching .txt label, and every label has an image.')
else:
    print(f'Found {len(problems)} unmatched file(s):')
    display(problems)

OK: every .jpg image has a matching .txt label, and every label has an image.


In [7]:
# Optional: save every file's matching status for later inspection.
output_path = Path('image_label_pair_report.csv')
pair_report.to_csv(output_path, index=False)
print(f'Saved report to: {output_path.resolve()}')

Saved report to: /home/mehedinaeem/Desktop/Code/Bitol_Computer_Vision_System/notebooks/image_label_pair_report.csv
